# 1 Intro into torch

## 1.1 Tensors

In [2]:
import torch
import numpy as np

Let's start with the basics. We create some dummy dataset: two observations $n$, every observation has three features $m$. We organize that as a dataset with dimensions $(n,m)$, so that is $(2,3)$

In [3]:
data = [
    [1, 2, 3],
    [10, 20, 30]
]

Our datatype here is `List[int]`, and PyTorch uses a `torch.Tensor` datatype.

In [4]:
X = torch.tensor(data)
type(X)

torch.Tensor

We can retrieve the shape

In [5]:
X.shape

torch.Size([2, 3])

And the type of the data inside the tensor:

In [6]:
X.dtype

torch.int64

Or the amount of observations:

In [7]:
len(X)

2

We can also start with a `numpy.array`

In [8]:
npdata = np.array(
    data,
    dtype = np.float32
)

Note we changed the dataformat to `np.float32`

In [9]:
X2 = torch.from_numpy(npdata)
X2

tensor([[ 1.,  2.,  3.],
        [10., 20., 30.]])

In [10]:
X2.dtype

torch.float32

## 1.2 Usefull functions for creating tensors

We can easily create a stand in tensor, with the same shape as our data:

In [11]:
ones = torch.ones_like(X2)
ones

tensor([[1., 1., 1.],
        [1., 1., 1.]])

Or random weights. These are uniform distributed positive numbers between 0 and 1

In [12]:
X3 = torch.rand(2,3)
X3

tensor([[0.3112, 0.2088, 0.7286],
        [0.8407, 0.8948, 0.2110]])

If we want normally distributed numbers, we need to specify mean and standard deviation:

In [13]:
X4 = torch.normal(mean=0.0, std=0.1, size=(2,3))
X4

tensor([[-0.1085,  0.0199, -0.0084],
        [ 0.1225, -0.2278,  0.1411]])

If your laptop or server has a GPU, PyTorch can run the calculations on the GPU. You can check if the GPU can be found by PyTorch with:

In [14]:
torch.cuda.is_available()

False

And you can set the tensor to the GPU device with `.to()`. Default is `"cpu"`

In [15]:
if torch.cuda.is_available():
    tensor = X3.to("cuda")
else:
    print("cuda not found")
X3.device

cuda not found


device(type='cpu')

For people with a macbook with an `mps` backend, there is mps acceleration available.

In [16]:
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
else:
    device = "cpu"
print(f"Using device {device}")
tensor = X3.to(device)
tensor

Using device cpu


tensor([[0.3112, 0.2088, 0.7286],
        [0.8407, 0.8948, 0.2110]])

Please note that using accelaration with cuda or mps is not always faster!
Reasons why this can be slower are:
- Memory transer: data needs to be transfered from cpu to gpu. This can be a bottleneck.
- parallel processing limits: some architectures (especially the RNNs we will learn about in lesson 3) cant be parallelized. 
- synchronisation overhead: running things in parallel also takes some overhead to synchronise the calculations, like waiting things to finish, merging them back together, etc.

This will especially be true for the simplere models and datasets we are using in the contexts of our lessons.

Other usefull tricks are to create an array of ones. Can you figure out how to create an array of zeros for yourself?

In [17]:
# Will try it here immediately.
zeroes = torch.zeros(1,10)
zeroes

# Well it looks like it worked!

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [18]:
ones = torch.ones(1, 10)
ones

tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])

Tensors can be concatenated. We need to specify the dimension along which the concatenation is done:

In [19]:
torch.cat([ones, ones, ones], dim=0)

tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])

## 1.3 Manipulation of tensors

The basis of most machine learning functions is the linear function. We can easily scale this by using matrix multiplication. Let's say we start with some random data, 32 observations with 10 features.

In [20]:
X = torch.rand(32, 10)

Now, if we want a linear map that transforms these 10 features into 2 dimensions, we can do that with a set of weights with dimensions $(10,2)$

In [21]:
W = torch.rand(10, 2)

In [22]:
yhat = X @ W
yhat.shape

torch.Size([32, 2])

Equivalent is this syntax:

In [23]:
yhat = torch.matmul(X, W)
yhat.shape

torch.Size([32, 2])

Torch will scale this up if you have more dimensions:

In [24]:
X = torch.rand(32, 10, 16)
W = torch.rand(16, 2)
yhat = X @ W
yhat.shape

torch.Size([32, 10, 2])

In [26]:
# Added by me (Max)

yhat

tensor([[[4.7577, 5.5239],
         [4.2690, 4.9829],
         [4.5872, 5.2891],
         [4.9958, 4.6175],
         [4.1834, 4.7691],
         [5.2886, 5.7435],
         [5.5021, 5.9457],
         [5.0283, 4.8176],
         [5.0996, 5.1175],
         [4.2295, 4.4706]],

        [[5.4915, 5.5582],
         [4.4550, 4.2864],
         [3.6795, 4.0190],
         [4.4043, 5.0188],
         [5.3639, 6.0756],
         [2.7215, 3.4431],
         [3.1213, 3.4082],
         [3.4514, 3.8316],
         [4.5603, 4.6211],
         [4.4343, 4.1291]],

        [[5.5770, 5.5413],
         [3.3070, 3.9309],
         [5.3444, 5.4795],
         [4.2999, 4.0667],
         [4.5736, 5.3084],
         [4.4513, 5.0749],
         [4.4505, 4.2330],
         [6.0760, 6.1034],
         [4.9343, 4.8767],
         [4.2296, 4.9954]],

        [[4.8418, 4.8748],
         [5.7165, 5.9002],
         [3.8866, 4.6181],
         [5.3003, 5.8190],
         [3.7272, 3.8980],
         [5.0689, 4.9570],
         [4.2822, 4.87

And finally, we can aggregate the tensor along the two features by taking the mean over the last dimension.

In [28]:
aggregate = yhat.mean(dim=-1)
aggregate.shape

torch.Size([32, 10])

In [29]:
# Comment Max

aggregate

tensor([[5.1408, 4.6259, 4.9381, 4.8067, 4.4763, 5.5160, 5.7239, 4.9229, 5.1086,
         4.3500],
        [5.5248, 4.3707, 3.8492, 4.7115, 5.7197, 3.0823, 3.2647, 3.6415, 4.5907,
         4.2817],
        [5.5591, 3.6190, 5.4119, 4.1833, 4.9410, 4.7631, 4.3418, 6.0897, 4.9055,
         4.6125],
        [4.8583, 5.8084, 4.2523, 5.5596, 3.8126, 5.0129, 4.5794, 4.7812, 4.4612,
         3.8904],
        [4.1866, 5.8970, 4.7611, 5.6047, 5.1705, 6.1726, 5.0059, 4.2393, 4.0230,
         5.7514],
        [5.1202, 4.5965, 3.9702, 3.5587, 5.0017, 5.6777, 4.6431, 3.8912, 4.9839,
         4.7571],
        [3.6548, 4.7363, 5.1823, 4.2476, 4.6281, 4.4324, 4.2408, 4.8125, 4.0651,
         4.4051],
        [5.9686, 4.4806, 4.3107, 4.7113, 5.0517, 4.1402, 4.0493, 5.2204, 4.9787,
         3.8651],
        [6.1000, 5.2025, 3.8548, 3.5120, 4.2722, 5.2127, 5.5934, 5.5836, 5.2629,
         5.3006],
        [4.0368, 5.1109, 3.3524, 5.2462, 5.3034, 4.0956, 3.1244, 5.1296, 5.2399,
         3.3690],
        [6

Try for yourself to calculate the sum

In [30]:
# Ok, will do. same principle but now with sum
aggregate_sum = yhat.sum(dim=-1)
aggregate_sum.shape

torch.Size([32, 10])

In [31]:
aggregate_sum

tensor([[10.2816,  9.2519,  9.8763,  9.6133,  8.9525, 11.0321, 11.4478,  9.8459,
         10.2171,  8.7000],
        [11.0497,  8.7414,  7.6985,  9.4230, 11.4395,  6.1646,  6.5295,  7.2830,
          9.1814,  8.5634],
        [11.1183,  7.2379, 10.8239,  8.3666,  9.8820,  9.5261,  8.6836, 12.1795,
          9.8110,  9.2249],
        [ 9.7166, 11.6168,  8.5047, 11.1193,  7.6252, 10.0259,  9.1587,  9.5624,
          8.9224,  7.7808],
        [ 8.3731, 11.7939,  9.5221, 11.2093, 10.3409, 12.3452, 10.0118,  8.4787,
          8.0459, 11.5028],
        [10.2405,  9.1929,  7.9404,  7.1175, 10.0034, 11.3555,  9.2862,  7.7825,
          9.9677,  9.5143],
        [ 7.3095,  9.4726, 10.3647,  8.4951,  9.2563,  8.8648,  8.4817,  9.6250,
          8.1302,  8.8103],
        [11.9372,  8.9612,  8.6214,  9.4227, 10.1033,  8.2804,  8.0986, 10.4407,
          9.9574,  7.7301],
        [12.1999, 10.4050,  7.7095,  7.0239,  8.5443, 10.4254, 11.1867, 11.1673,
         10.5258, 10.6012],
        [ 8.0736, 1

## 1.4 GPU or CPU

Tensors live in the CPU or GPU:

In [32]:
X.device

device(type='cpu')

You can check if you have a GPU available:

In [33]:
torch.cuda.is_available()

False

Or a mac with M1

In [34]:
torch.backends.mps.is_available()

False

And move a tensor to the GPU for faster computing, if available

In [35]:
if torch.cuda.is_available():
    X_ = X.to("cuda")
elif torch.backends.mps.is_available():
    X_ = X.to("mps")
else:
    X_ = X.to("cpu")
X_.device

device(type='cpu')

## 1.5 Reshape or View

Often, you will need to reshape a tensor:

In [36]:
X = torch.rand(32, 28, 28, 1)
X_view = X.view(32, 28*28)
X_reshape = X.reshape(32, 28*28)
X.shape, X_view.shape, X_reshape.shape

(torch.Size([32, 28, 28, 1]), torch.Size([32, 784]), torch.Size([32, 784]))

The difference between `view` and `reshape` is: `view` operates as a view on the original tensor. If the underlying data is changed, the view will change too.

No data movement occurs when creating a view, view tensor just changes the way it interprets the same data.

In [37]:
X = torch.Tensor([0, 0])
X_view = X.view(1,2)
X.storage().data_ptr() == X_view.storage().data_ptr()

C:\Users\D110303\AppData\Local\Temp\ipykernel_22612\3787215844.py:3: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  X.storage().data_ptr() == X_view.storage().data_ptr()


True

In [38]:
X[0] = 1
X_view

tensor([[1., 0.]])

`view` can throw an error if the required view is not contiguous (does not share the same memory block)

> A tensor whose values are laid out in the storage starting from the rightmost dimension onward (that is, moving along rows for a 2D tensor) is defined as contiguous. Contiguous tensors are convenient because we can visit them efficiently in order without jumping around in the storage (improving data locality improves performance because of the way memory access works on modern CPUs). This advantage of course depends on the way algorithms visit.

You could call `.contiugous()` on a `view`, but `.reshape()` does that behind the scenes.

## 1.6 Permute

Sometimes you might want to reshuffle the order of a tensor.

For example, let's say we load an batch of 32 images, where every image has a size of 28x28 pixels, and has 3 channels (RGB color)

In [39]:
X = torch.rand(32, 28, 28, 3)

In [47]:
# Comment Max to better understand what is happening
X

tensor([[[[6.9427e-02, 3.3122e-01, 4.4087e-01],
          [4.9630e-01, 1.8208e-01, 8.2150e-01],
          [6.5464e-01, 5.6217e-01, 9.4861e-01],
          ...,
          [7.1943e-01, 2.4025e-01, 7.2801e-01],
          [9.0763e-01, 9.9172e-01, 4.7325e-02],
          [2.4413e-01, 7.0194e-02, 5.6062e-01]],

         [[4.7464e-01, 7.8838e-01, 3.2366e-01],
          [9.6121e-01, 6.7965e-01, 1.7226e-01],
          [1.0096e-01, 2.9448e-01, 4.8693e-01],
          ...,
          [6.4602e-01, 6.0513e-01, 3.2348e-01],
          [6.7284e-01, 5.1619e-01, 8.5632e-01],
          [4.4439e-01, 1.5888e-01, 3.5034e-02]],

         [[3.7598e-01, 8.9743e-01, 1.1011e-01],
          [5.6776e-01, 1.0333e-01, 4.5067e-02],
          [2.1234e-01, 9.4015e-01, 2.7188e-01],
          ...,
          [8.0000e-01, 5.5935e-01, 8.7839e-01],
          [9.2386e-01, 4.0182e-01, 2.9934e-01],
          [7.2376e-01, 1.8322e-01, 7.1620e-01]],

         ...,

         [[7.1430e-01, 8.9358e-01, 1.7731e-01],
          [7.7323e-01,

It is the case that there are different conventions for manipulating tensors in image recognition models. Some models have a channel-last convention, like I used above, but some (like [pytorch](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)) use a channel first convention, which would be (batch, channel, height, width).

You would want to swap the 4th dimension to the 2nd, or if you start from zero:

In [40]:
channel_first = X.permute(0, 3, 1, 2)
channel_first.shape

torch.Size([32, 3, 28, 28])

## 1.7 Broadcasting

Broadcasting is something you might know from `numpy`, but it is also used by `tensorflow`, `jax` and `torch`. 

Broadcasting allows to extend a dimension, without the need to do so explicitly. The rules for broadcasting are simple:

- two dimesions are equal
- one of the dimensions is 1
- one of the dimensions does not exist

but lets show an example

In [41]:
a = torch.ones(2, 2)
b = torch.ones(2, 2)
a, b, a+b

(tensor([[1., 1.],
         [1., 1.]]),
 tensor([[1., 1.],
         [1., 1.]]),
 tensor([[2., 2.],
         [2., 2.]]))

This is straigh forward. But what would happen in this case:

In [42]:
a = torch.ones(1, 2)
b = torch.ones(2, 2)

`b` is a 2x2 grid, and has four numbers. If we want to add `a`, we have only two numbers! Now, you could start stacking the `a` tensor to get matching dimensions. But you dont have to!

In [43]:
a + b

tensor([[2., 2.],
        [2., 2.]])

See what happened here? 

`a` is magically broadcasted over the first dimension. And what would you guess would happen in this case:

In [49]:
a = torch.ones(1, 5, 1, 4)
b = torch.ones(3, 1, 3, 1)

In [50]:
# Comment Max
(a+b).shape

torch.Size([3, 5, 3, 4])

First, predict the output shape, then check it for yourself.

And, what would you think happens here; do you think this gives an error, or do you think it broadcasts?

In [51]:
a = torch.ones(5, 1, 4)
b = torch.ones(3, 1, 3, 1)

In [52]:
(a+b).shape

torch.Size([3, 5, 3, 4])